# Day 37: Feature Importance Methods & Interpretability
**Author:** Sahil-K-Y  
**Phase:** 3 - Tree Models & SVM  
**Date:** Day 037

---

## 1. The Value of Explainable AI (XAI)

In real-world applications, understanding **why** a model makes a prediction is often as critical as the prediction itself. 
* **Trust:** Medical clinicians or financial loan officers are unlikely to adopt a "black-box" model.
* **Feature Selection:** Removing irrelevant features simplifies models, reduces training time, and minimizes overfitting.
* **Debugging:** Identifying if a model relies on spurious correlations (e.g., classifying a wolf based on snow in the background).
* **Business Insights:** Discovering key drivers of user churn, risk, or medical diagnoses.


## 2. Three Feature Importance Methodologies

We will study three ways of calculating feature importance:

### A. Gini Importance (Mean Decrease in Impurity - MDI)
* **How it works:** During tree construction, whenever a node splits on a feature, the impurity of the child nodes decreases. Gini Importance calculates the sum of all Gini impurity decreases (weighted by the fraction of samples reaching the split node) across all trees in the forest.
* **Pros:** Extremely fast to compute. It is computed during training and is available out-of-the-box.
* **Cons:** Heavily biased toward high-cardinality features (features containing many unique numerical values or categories), as they offer more opportunities for splitting.

### B. Permutation Importance (Mean Decrease in Accuracy - MDA)
* **How it works:** 
  1. Train a model and compute score on validation data.
  2. For a specific feature, randomly shuffle its values across the validation set (decoupling the link between the feature and the target while keeping the dataset distribution intact).
  3. Re-evaluate the model's accuracy on the shuffled validation set. The drop in accuracy is the feature's importance.
* **Pros:** Model-agnostic (works on SVM, Linear models, etc.), unbiased toward feature cardinality.
* **Cons:** Computationally slower; if two features are highly correlated, shuffling one might not drop accuracy much because the model can reconstruct the information from the other feature.

### C. Drop-Column Importance
* **How it works:** 
  1. Train model on all features and measure accuracy.
  2. For each feature: drop it entirely, retrain a new model, and measure accuracy. The difference in accuracy is the feature's true contribution.
* **Pros:** The gold standard of feature importance.
* **Cons:** Brutally slow. Requires training $N+1$ independent models, where $N$ is the number of features.


## Exercise 1: Gini-based Feature Importance on Wine

Let's extract and plot Gini importance on the Wine dataset.

### Step 1.1: Load Dataset
Let's import `load_wine` and prepare our train-test split.


In [ ]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split

wine = load_wine()
X, y = wine.data, wine.target
feature_names = wine.feature_names

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
print("Wine dataset loaded and partitioned.")


### Step 1.2: Fit Random Forest
Let's train our classifier on the training set.


In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_wine = RandomForestClassifier(n_estimators=100, random_state=42)
rf_wine.fit(X_train, y_train)
print("Baseline model trained.")


### Step 1.3: Extract and Plot Gini Importance
Let's extract the `feature_importances_` attribute.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

gini_importances = rf_wine.feature_importances_

df_gini = pd.DataFrame({
    'Feature': feature_names,
    'Gini_Importance': gini_importances
}).sort_values(by='Gini_Importance', ascending=False)

sns.barplot(x='Gini_Importance', y='Feature', data=df_gini, palette='viridis')
plt.title('Gini Importance (Mean Decrease in Impurity) - Wine Dataset', fontsize=14, fontweight='bold')
plt.xlabel('Importance Score')
plt.show()


## Exercise 2: Permutation Importance on Wine

Let's compute permutation importance on the test set and compare it to Gini importance.

### Step 2.1: Run Permutation Importance
We'll import `permutation_importance` and run it with `n_repeats=10`.


In [ ]:
from sklearn.inspection import permutation_importance

perm_result = permutation_importance(
    rf_wine, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1
)

df_perm = pd.DataFrame({
    'Feature': feature_names,
    'Perm_Importance_Mean': perm_result.importances_mean,
    'Perm_Importance_Std': perm_result.importances_std
}).sort_values(by='Perm_Importance_Mean', ascending=False)

print(df_perm.head())


### Step 2.2: Plot Permutation Importances
We render it showing error bars representing the standard deviation across shuffles.


In [ ]:
plt.errorbar(x=df_perm['Perm_Importance_Mean'], y=df_perm['Feature'], 
             xerr=df_perm['Perm_Importance_Std'], fmt='o', color='darkblue', ecolor='lightblue', elinewidth=3, capsize=0)
plt.title('Permutation Importance (Mean Decrease in Accuracy) - Wine Dataset', fontsize=14, fontweight='bold')
plt.xlabel('Mean Drop in Test Accuracy')
plt.show()


## Exercise 3: Feature Selection Sweep

Let's evaluate how our classification accuracy changes as we train models using only the top $N$ features (ordered by Gini importance).

### Step 3.1: Run Selection Loop
We write a loop to incrementally evaluate performance as we add more features.


In [ ]:
from sklearn.metrics import accuracy_score

sorted_features_gini = df_gini['Feature'].tolist()

X_train_df = pd.DataFrame(X_train, columns=feature_names)
X_test_df = pd.DataFrame(X_test, columns=feature_names)

accuracies = []

for n in range(1, len(feature_names) + 1):
    selected_features = sorted_features_gini[:n]
    
    X_tr_sliced = X_train_df[selected_features]
    X_te_sliced = X_test_df[selected_features]
    
    clf = RandomForestClassifier(n_estimators=100, random_state=42)
    clf.fit(X_tr_sliced, y_train)
    acc = accuracy_score(y_test, clf.predict(X_te_sliced))
    accuracies.append(acc)

print("Selection sweep completed.")


### Step 3.2: Plot Selection Performance
Let's visualize accuracy as a function of feature count.


In [ ]:
plt.plot(range(1, len(feature_names) + 1), accuracies, 'o-', color='darkgreen', linewidth=2)
plt.title('Model Accuracy vs. Number of Top Features Used', fontsize=14, fontweight='bold')
plt.xlabel('Number of Top Features (Gini)')
plt.ylabel('Test Accuracy')
plt.xticks(range(1, len(feature_names) + 1))
plt.grid(True)
plt.show()


## Exercise 4: Demonstration of Shared Importance via Correlated Features

When two features are highly correlated, tree-based models split their importance between them. Shuffling or dropping one doesn't significantly drop accuracy because the model can rely on the other. Let's demonstrate this.

### Step 4.1: Retrieve Baseline Importances
Let's see Flavanoids vs. Total Phenols correlation and importances.


In [ ]:
correlation = X_train_df['flavanoids'].corr(X_train_df['total_phenols'])
print(f"Correlation: {correlation:.4f}")

base_flav_imp = rf_wine.feature_importances_[feature_names.index('flavanoids')]
base_tp_imp = rf_wine.feature_importances_[feature_names.index('total_phenols')]

print(f"Baseline Flavanoids Importance: {base_flav_imp:.4%}")
print(f"Baseline Total Phenols Importance: {base_tp_imp:.4%}")


### Step 4.2: Drop One Feature and Retrain
We drop `flavanoids` and see if the other absorbs the shared importance.


In [ ]:
rf_no_flav = RandomForestClassifier(n_estimators=100, random_state=42)
rf_no_flav.fit(X_train_df.drop(columns=['flavanoids']), y_train)

new_tp_index = X_train_df.drop(columns=['flavanoids']).columns.tolist().index('total_phenols')
new_tp_imp = rf_no_flav.feature_importances_[new_tp_index]

print(f"New Total Phenols Importance: {new_tp_imp:.4%}")
print(f"Absolute Lift: {new_tp_imp - base_tp_imp:+.4%}")


## Exercise 5: Three-Method Comparison on Breast Cancer Dataset

We will compute and plot Gini, Permutation, and Drop-Column importances on the **Breast Cancer Dataset** to compare the rankings.

### Step 5.1: Initialize Breast Cancer Dataset
Let's load the cancer data.


In [ ]:
from sklearn.datasets import load_breast_cancer

bc = load_breast_cancer()
X_bc, y_bc = bc.data, bc.target
fn_bc = bc.feature_names.tolist()

X_bc_tr, X_bc_te, y_bc_tr, y_bc_te = train_test_split(
    X_bc, y_bc, test_size=0.3, random_state=42, stratify=y_bc
)

df_bc_tr = pd.DataFrame(X_bc_tr, columns=fn_bc)
df_bc_te = pd.DataFrame(X_bc_te, columns=fn_bc)

rf_bc = RandomForestClassifier(n_estimators=100, random_state=42)
rf_bc.fit(df_bc_tr, y_bc_tr)
base_bc_acc = accuracy_score(y_bc_te, rf_bc.predict(df_bc_te))
print("Breast cancer baseline model ready.")


### Step 5.2: Compute Gini and Permutation Importances
Let's extract Gini and calculate Permutation mean values.


In [ ]:
gini_bc = rf_bc.feature_importances_

perm_bc_res = permutation_importance(rf_bc, df_bc_te, y_bc_te, n_repeats=5, random_state=42, n_jobs=-1)
perm_bc = perm_bc_res.importances_mean
print("Gini and Permutation importances computed.")


### Step 5.3: Compute Drop-Column Importance
Since drop-column requires retraining $N$ models, we'll only run it for the top 10 features sorted by Gini.


In [ ]:
import numpy as np

drop_col_bc = np.zeros(len(fn_bc))
top_10_gini_idx = np.argsort(gini_bc)[::-1][:10]

for i in top_10_gini_idx:
    feat = fn_bc[i]
    X_tr_d = df_bc_tr.drop(columns=[feat])
    X_te_d = df_bc_te.drop(columns=[feat])
    
    rf_d = RandomForestClassifier(n_estimators=100, random_state=42)
    rf_d.fit(X_tr_d, y_bc_tr)
    drop_acc = accuracy_score(y_bc_te, rf_d.predict(X_te_d))
    drop_col_bc[i] = base_bc_acc - drop_acc

print("Drop column calculation finished.")


### Step 5.4: Side-by-Side Plotting of Three Methods
Let's construct a triple-bar chart comparing the importances side-by-side.


In [ ]:
df_bc_comparison = pd.DataFrame({
    'Feature': fn_bc,
    'Gini': gini_bc,
    'Permutation': perm_bc,
    'Drop_Column': drop_col_bc
})

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

top_g = df_bc_comparison.sort_values(by='Gini', ascending=False).head(10)
sns.barplot(x='Gini', y='Feature', data=top_g, ax=axes[0], palette='Blues_r')
axes[0].set_title('Top 10 Gini Importance')

top_p = df_bc_comparison.sort_values(by='Permutation', ascending=False).head(10)
sns.barplot(x='Permutation', y='Feature', data=top_p, ax=axes[1], palette='Oranges_r')
axes[1].set_title('Top 10 Permutation Importance')

top_d = df_bc_comparison.sort_values(by='Drop_Column', ascending=False).head(10)
sns.barplot(x='Drop_Column', y='Feature', data=top_d, ax=axes[2], palette='Greens_r')
axes[2].set_title('Top 10 Drop-Column Importance')

plt.tight_layout()
plt.show()
